# Week 11: Testing and classroom delivery — the case

**Start here:** Version 2 answers faster but exposes one staff-only policy to a student. Would you release it?

**Your goal:** Record a safe release decision with test and rollback evidence.

Run the prepared cells from top to bottom. Predict the result before changing one input. These small demonstrations isolate an idea; they are not trained production models.

**Concepts to point to:** Release gate, Filtered trace, Rollback, Cost per success.

# Week 11: test failures and decide a classroom release

These checks call the actual local starter. No cloud deployment occurs.

In [ ]:
import re
POLICIES = [{'id': 'P1', 'text': 'Student travel costs including bus tickets are not reimbursed.', 'role': 'student', 'version': '2027-01'}, {'id': 'P2', 'text': 'Library loans may be extended once for seven days.', 'role': 'student', 'version': '2027-01'}, {'id': 'P3', 'text': 'Students may request one deadline extension within seven days of the original due date.', 'role': 'student', 'version': '2027-01'}, {'id': 'P4', 'text': 'Academic appeals must be filed within ten days of the result.', 'role': 'student', 'version': '2027-01'}, {'id': 'P5', 'text': 'Staff payroll changes require manager approval before any record is changed.', 'role': 'staff', 'version': '2027-01'}]
STOP={'a','an','the','is','are','to','of','my','can','i','what','how','for','in','and'}
def tokens(text):return set(re.findall(r'[a-z]+',text.lower()))-STOP
def rank(question,role='student',k=1,corpus=None):
    if not isinstance(question,str) or not question.strip():raise ValueError('Question must be nonempty text')
    if role not in ('student','staff'):raise ValueError('Unknown trusted role')
    if k not in (1,2):raise ValueError('k must be 1 or 2')
    corpus=POLICIES if corpus is None else corpus
    allowed=[p for p in corpus if p['role']=='student' or role=='staff']
    scored=sorted([(len(tokens(question)&tokens(p['text'])),p) for p in allowed],key=lambda x:x[0],reverse=True)
    return [p for score,p in scored if score>=2][:k]
def answer(question,role='student',k=1,corpus=None):
    found=rank(question,role,k,corpus)
    return {'status':'extract' if found else 'unsupported','text':' '.join(p['text'] for p in found),
            'sources':[p['id'] for p in found],'version':'2027-01'}
def safe_trace(request_id,result):
    return {'request_id':request_id,'status':result['status'],'sources':result['sources'],'version':result['version']}


## Run invalid-input, missing-source and access checks

In [ ]:
checks={}
try:answer('')
except ValueError:checks['invalid_input_blocked']=True
else:checks['invalid_input_blocked']=False
checks['missing_source_abstains']=answer('academic appeals filed',corpus=[])['status']=='unsupported'
checks['staff_access_blocked']='P5' not in answer('staff payroll changes','student')['sources']
print(checks)
assert all(checks.values())
trace=safe_trace('request-01',answer('academic appeals filed'))
print('Privacy-filtered trace:',trace)
assert 'question' not in trace and 'text' not in trace


## Candidate regression and rollback
The candidate is a deliberate faulty function. A test discovers the failure; the result is not hardcoded.

In [ ]:
def faulty_candidate(question,role='student'):
    return answer(question,'staff')  # deliberately ignores the user's role
active=answer
candidate_allowed='P5' not in faulty_candidate('staff payroll changes','student')['sources']
if candidate_allowed:active=faulty_candidate
print('candidate allowed:',candidate_allowed,'active:',active.__name__)
assert active is answer
cost=4;successes=8
print('cost per successful task:',cost/successes if successes else None)


## Explain and transfer

1. What did you change, and what happened? Flip the candidate role check and verify rollback happens.
2. Which diagram block produced the difference?
3. What is one limitation of the demonstration?
4. Fresh situation: The new version passes answer checks but fails the missing-source test. What does the release gate decide?

**Evidence to save:** Filtered trace and classroom release checklist.